In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [ ]:
G = 1.0  

def accelerations(positions, masses):
    """
    positions: (N, 2) массив координат [x, y]
    masses: (N,) массив масс
    return: (N, 2) ускорения
    """
    N = len(masses)
    acc = np.zeros_like(positions)
    for i in range(N):
        for j in range(N):
            if i != j:
                r_vec = positions[j] - positions[i]
                r = np.linalg.norm(r_vec)
                acc[i] += G * masses[j] * r_vec / r**3
    return acc



def rk4_step(state, dt, masses):
    N = len(masses)
    def f(state):
        pos = state[:N]
        vel = state[N:]
        acc = accelerations(pos, masses)
        return np.vstack([vel, acc])

    k1 = f(state)
    k2 = f(state + 0.5 * dt * k1)
    k3 = f(state + 0.5 * dt * k2)
    k4 = f(state + dt * k3)

    return state + (dt/6.0) * (k1 + 2*k2 + 2*k3 + k4)


def simulate(method_step, dt, t_end, masses, pos0, vel0):
    """
    pos0: (N,2) начальные координаты
    vel0: (N,2) начальные скорости
    """
    N = len(masses)
    steps = int(np.floor(t_end/dt)) + 1
    t = np.array([i*dt for i in range(steps)])

    # состояние: (2N, 2), сначала позиции, потом скорости
    state = np.vstack([pos0, vel0])

    positions = np.zeros((steps, N, 2))
    velocities = np.zeros((steps, N, 2))

    for i in range(steps):
        positions[i] = state[:N]
        velocities[i] = state[N:]
        state = method_step(state, dt, masses)

    return positions, velocities, t


Моделируем движение, похожее на движение Солнца, Земли, Луны.

In [ ]:
G = 1.0

M_sun   = 333000.
M_earth = 1.0
M_moon  = 1.0 / 81.3   

masses = np.array([M_sun, M_earth, M_moon], dtype=float)

r_SE = 10.0    
r_EM = 0.3  

pos_sun   = np.array([0.0, 0.0])
pos_earth = np.array([r_SE, 0.0])
pos_moon  = pos_earth + np.array([r_EM, 0.0])

pos0 = np.vstack([pos_sun, pos_earth, pos_moon])


v_earth = np.array([0.0, math.sqrt(G * M_sun / r_SE)])
v_moon_rel = np.array([0.0, math.sqrt(G * M_earth / r_EM)])
v_moon = v_earth + v_moon_rel

v_sun = np.array([0.0, 0.0])

vel0 = np.vstack([v_sun, v_earth, v_moon])

positions, velocities, t = simulate(rk4_step, dt=0.001, t_end=1, masses=masses, pos0=pos0, vel0=vel0)
speed = np.linalg.norm(velocities, axis=2)

In [ ]:
import matplotlib as mpl
from IPython.display import HTML

mpl.rcParams['animation.embed_limit'] = 50  

N = len(masses)


max_x = np.max(positions[:,:,0])
min_x = np.min(positions[:,:,0])
max_y = np.max(positions[:,:,1])
min_y = np.min(positions[:,:,1])

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(min_x*1.2, max_x*1.2)
ax.set_ylim(min_y*1.2, max_y*1.2)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Траектории трёх тел")
ax.grid(True)

lines = [ax.plot([], [], '-', lw=1.5)[0] for _ in range(N)]
points = [ax.plot([], [], 'o', ms=8)[0] for _ in range(N)]
labels = [ax.text(0,0,"", fontsize=9) for _ in range(N)]

max_len = len(t)

def init():
    for line, point, label in zip(lines, points, labels):
        line.set_data([], [])
        point.set_data([], [])
        label.set_text("")
    return lines + points + labels

def animate(frame):
    for i in range(N):
        xs = positions[:frame, i, 0]
        ys = positions[:frame, i, 1]
        lines[i].set_data(xs, ys)
        if frame < len(xs):
            points[i].set_data([positions[frame, i, 0]], [positions[frame, i, 1]])
            labels[i].set_position((positions[frame, i, 0], positions[frame, i, 1]))
            labels[i].set_text(f"Body {i+1}")
        else:
            points[i].set_data([], [])
    return lines + points + labels

anim = animation.FuncAnimation(fig, animate, init_func=init,
                               frames=max_len, interval=20, blit=True)

HTML(anim.to_jshtml())

Непохоже на движение Земли и Луны вокруг Солнца. Больше похоже на движение 2 планет вокруг Солнца. Нарисуем графики скоростей.

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(t, speed[:, 0], label=f'Солнце')
plt.xlabel("Time")
plt.ylabel("Speed")
plt.title("Модуль скорости тел от времени")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(t, speed[:, 1], label=f'Земля')
plt.xlabel("Time")
plt.ylabel("Speed")
plt.title("Модуль скорости тел от времени")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(t, speed[:, 2], label=f'Луна')
plt.xlabel("Time")
plt.ylabel("Speed")
plt.title("Модуль скорости тел от времени")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
G = 1.0

M_sun   = 1000.
M_earth = 1.0
M_moon  = 1.0 / 81.3   

masses = np.array([M_sun, M_earth, M_moon], dtype=float)

r_SE = 10.0    
r_EM = 0.3  

pos_sun   = np.array([0.0, 0.0])
pos_earth = np.array([r_SE, 0.0])
pos_moon  = pos_earth + np.array([r_EM, 0.0])

pos0 = np.vstack([pos_sun, pos_earth, pos_moon])


v_earth = np.array([0.0, math.sqrt(G * M_sun / r_SE)])
v_moon_rel = np.array([0.0, math.sqrt(G * M_earth / r_EM)])
v_moon = v_earth + v_moon_rel

v_sun = np.array([0.0, 0.0])

vel0 = np.vstack([v_sun, v_earth, v_moon])

positions, velocities, t = simulate(rk4_step, dt=0.01, t_end=8, masses=masses, pos0=pos0, vel0=vel0)
speed = np.linalg.norm(velocities, axis=2)

In [ ]:
import matplotlib as mpl
from IPython.display import HTML

mpl.rcParams['animation.embed_limit'] = 100  

N = len(masses)


max_x = np.max(positions[:,:,0])
min_x = np.min(positions[:,:,0])
max_y = np.max(positions[:,:,1])
min_y = np.min(positions[:,:,1])

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(min_x*1.2, max_x*1.2)
ax.set_ylim(min_y*1.2, max_y*1.2)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Траектории трёх тел")
ax.grid(True)

lines = [ax.plot([], [], '-', lw=1.5)[0] for _ in range(N)]
points = [ax.plot([], [], 'o', ms=8)[0] for _ in range(N)]
labels = [ax.text(0,0,"", fontsize=9) for _ in range(N)]

max_len = len(t)

def init():
    for line, point, label in zip(lines, points, labels):
        line.set_data([], [])
        point.set_data([], [])
        label.set_text("")
    return lines + points + labels

def animate(frame):
    for i in range(N):
        xs = positions[:frame, i, 0]
        ys = positions[:frame, i, 1]
        lines[i].set_data(xs, ys)
        if frame < len(xs):
            points[i].set_data([positions[frame, i, 0]], [positions[frame, i, 1]])
            labels[i].set_position((positions[frame, i, 0], positions[frame, i, 1]))
            labels[i].set_text(f"Body {i+1}")
        else:
            points[i].set_data([], [])
    return lines + points + labels

anim = animation.FuncAnimation(fig, animate, init_func=init,
                               frames=max_len, interval=20, blit=True)

HTML(anim.to_jshtml())

Это уже похоже на Солнце, Землю, Луну

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(t, speed[:, 0], label=f'Солнце')
plt.xlabel("Time")
plt.ylabel("Speed")
plt.title("Модуль скорости тел от времени")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(t, speed[:, 1], label=f'Земля')
plt.xlabel("Time")
plt.ylabel("Speed")
plt.title("Модуль скорости тел от времени")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(t, speed[:, 2], label=f'Луна')
plt.xlabel("Time")
plt.ylabel("Speed")
plt.title("Модуль скорости тел от времени")
plt.legend()
plt.grid(True)
plt.show()